# Diabetes model for TrustyAI SPD demo (AgeOver50)

This notebook rebuilds the diabetes demo model so that:

- the raw `Age` feature is replaced with a binary `AgeOver50`
- normalization is applied **inside the PyTorch model**
- the exported ONNX model accepts **raw feature values** at inference time
- `AgeOver50` remains a clean binary protected attribute for TrustyAI SPD

This avoids the common mismatch where a model is trained on scaled inputs but the deployed ONNX model receives raw inputs.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


In [ ]:
DATA_PATH = Path("diabetes-pytorch/pima-indians-diabetes.data.csv")
MODEL_PATH = Path("diabetes.onnx")
TRAINING_DATA_EXPORT = Path("diabetes-trustyai-training-data.csv")
SAMPLE_PAYLOAD_DIR = Path("sample-payloads")

RANDOM_STATE = 42
TEST_SIZE = 0.20
EPOCHS = 300
LEARNING_RATE = 0.005

# Demo-only option: flip a fraction of favorable labels for AgeOver50 == 1
ENABLE_SYNTHETIC_BIAS = False
SYNTHETIC_BIAS_FLIP_RATE = 0.20

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


## Load the dataset


In [ ]:
names = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

df = pd.read_csv(DATA_PATH, names=names)
print(df.head())
print()
print(df.describe(include="all"))


## Replace `Age` with `AgeOver50`


In [ ]:
df["AgeOver50"] = (df["Age"] > 50).astype(np.float32)
df = df.drop(columns=["Age"])

feature_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "AgeOver50",
]
target_col = "Outcome"

continuous_cols = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
]
binary_cols = ["AgeOver50"]

X_df = df[feature_cols].copy()
y = df[target_col].astype(np.int64).values

print("Feature columns:", feature_cols)
print()
print("AgeOver50 counts:")
print(df["AgeOver50"].value_counts().sort_index())


## Train/test split on **raw** inputs

We keep the dataframe values raw here.  
Normalization will be done inside the model, using statistics computed only from the training set.


In [ ]:
X = X_df.values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_df = pd.DataFrame(X_train, columns=feature_cols)
test_df = pd.DataFrame(X_test, columns=feature_cols)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)


## Compute normalization statistics from the training set

Only the 7 continuous columns are normalized.  
`AgeOver50` stays as a literal `0.0/1.0`.


In [ ]:
train_means = train_df[continuous_cols].mean().astype(np.float32)
train_stds = train_df[continuous_cols].std(ddof=0).replace(0, 1).astype(np.float32)

print("Training means:")
print(train_means)
print()
print("Training stds:")
print(train_stds)


## Optional synthetic bias for a clearer SPD demo

Leave this disabled for a natural model.  
Enable it only if you want a stronger visible SPD signal in a lab/demo.


In [ ]:
y_train_effective = y_train.copy()

if ENABLE_SYNTHETIC_BIAS:
    older_mask = train_df["AgeOver50"].values == 1.0
    favorable_mask = y_train_effective == 0

    candidate_idx = np.where(older_mask & favorable_mask)[0]
    rng = np.random.default_rng(RANDOM_STATE)
    flip_count = int(len(candidate_idx) * SYNTHETIC_BIAS_FLIP_RATE)

    if flip_count > 0:
        flip_idx = rng.choice(candidate_idx, size=flip_count, replace=False)
        y_train_effective[flip_idx] = 1

    print(f"Synthetic bias enabled. Flipped {flip_count} labels from 0 -> 1 for AgeOver50 == 1.")
else:
    print("Synthetic bias disabled.")


## Define the model with built-in normalization

This is the key fix.

The model accepts **raw feature values**.  
Inside `forward()`, it standardizes the continuous features using the stored training means/stds, leaves `AgeOver50` untouched, and then runs the dense network.

Because the normalization lives inside the PyTorch model, it is exported into ONNX too.


In [ ]:
class DiabetesModel(nn.Module):
    def __init__(self, means, stds):
        super().__init__()

        self.register_buffer("means", torch.tensor(means.values, dtype=torch.float32))
        self.register_buffer("stds", torch.tensor(stds.values, dtype=torch.float32))

        self.net = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )

    def preprocess(self, x):
        x_cont = x[:, :7]
        x_bin = x[:, 7:].clone()  # AgeOver50
        x_cont = (x_cont - self.means) / self.stds
        return torch.cat([x_cont, x_bin], dim=1)

    def forward(self, x):
        x = self.preprocess(x)
        logits = self.net(x)
        probs = torch.softmax(logits, dim=1)
        labels = torch.argmax(probs, dim=1)
        return labels, probs


model = DiabetesModel(train_means, train_stds)
model


## Train


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train_effective)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    logits = model.net(model.preprocess(X_train_t))
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 25 == 0 or epoch == 0:
        print(f"Epoch {epoch + 1:03d}/{EPOCHS} - loss: {loss.item():.4f}")


## Evaluate on the test set


In [ ]:
model.eval()

with torch.no_grad():
    X_test_t = torch.FloatTensor(X_test)
    pred_labels, pred_probs = model(X_test_t)

y_pred = pred_labels.numpy()
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(float(accuracy), 4))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))


## Quick raw-input smoke test

These are raw feature values in the same order you will send to the deployed model.

If the model is healthy, these outputs should not all be identical.


In [ ]:
raw_smoke = np.array([
    [0.0, 92.0, 68.0, 18.0, 80.0, 23.5, 0.18, 0.0],
    [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0],
    [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 1.0],
    [5.0, 185.0, 96.0, 35.0, 240.0, 40.0, 0.82, 1.0],
], dtype=np.float32)

with torch.no_grad():
    smoke_labels, smoke_probs = model(torch.tensor(raw_smoke))

print("Labels:", smoke_labels.numpy().tolist())
print("Probabilities:")
print(smoke_probs.numpy())


## Export ONNX

The exported model accepts **raw** inference payloads.


In [ ]:
dummy_input = torch.randn(1, len(feature_cols), dtype=torch.float32)

torch.onnx.export(
    model,
    dummy_input,
    MODEL_PATH.as_posix(),
    verbose=False,
    input_names=["dense_input"],
    output_names=["label", "probabilities"],
    dynamic_axes={
        "dense_input": {0: "batch_size"},
        "label": {0: "batch_size"},
        "probabilities": {0: "batch_size"},
    },
    opset_version=11,
)

print(f"Saved ONNX model to: {MODEL_PATH.resolve()}")


## Export training data for TrustyAI

This export uses the raw feature values, including `AgeOver50`, which is what you want TrustyAI to see.


In [ ]:
export_df = pd.DataFrame(X, columns=feature_cols)
export_df["Outcome"] = y

export_df.to_csv(TRAINING_DATA_EXPORT, index=False)
print(f"Saved training data export to: {TRAINING_DATA_EXPORT.resolve()}")
print(export_df.head())


## Save sample inference payloads


In [ ]:
SAMPLE_PAYLOAD_DIR.mkdir(parents=True, exist_ok=True)

sample_under_50 = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [1, 8],
            "datatype": "FP32",
            "data": [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0],
        }
    ],
}

sample_over_50 = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [1, 8],
            "datatype": "FP32",
            "data": [2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 1.0],
        }
    ],
}

sample_batch = {
    "model_name": "diabetes",
    "inputs": [
        {
            "name": "dense_input",
            "shape": [4, 8],
            "datatype": "FP32",
            "data": [
                0.0, 92.0, 68.0, 18.0, 80.0, 23.5, 0.18, 0.0,
                2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 0.0,
                2.0, 140.0, 80.0, 25.0, 150.0, 31.5, 0.45, 1.0,
                5.0, 185.0, 96.0, 35.0, 240.0, 40.0, 0.82, 1.0,
            ],
        }
    ],
}

for name, payload in {
    "under50.json": sample_under_50,
    "over50.json": sample_over_50,
    "batch.json": sample_batch,
}.items():
    p = SAMPLE_PAYLOAD_DIR / name
    p.write_text(json.dumps(payload, indent=2))
    print(f"Saved {p.resolve()}")


## Example commands

Use these after redeploying the new ONNX model.


In [ ]:
print(f'''
# Single infer request example
curl -sk \
  -H "Authorization: Bearer $TOKEN" \
  -H "Content-Type: application/json" \
  https://diabetes-myproj1.apps.ocp4.example.com/v2/models/diabetes/infer \
  -d '{json.dumps(sample_under_50)}' | jq

# TrustyAI info check
curl -sk -H "Authorization: Bearer $TOKEN" "$TRUSTY_ROUTE/info" | jq

# Example SPD request after new observations exist
curl -sk -H "Authorization: Bearer $TOKEN" \
  -H 'Content-Type: application/json' \
  -X POST "$TRUSTY_ROUTE/metrics/group/fairness/spd" \
  -d '{{
    "modelId": "diabetes",
    "protectedAttribute": "AgeOver50",
    "privilegedAttribute": {{"type":"FLOAT","value":0}},
    "unprivilegedAttribute": {{"type":"FLOAT","value":1}},
    "outcomeName": "Final_Prediction",
    "favorableOutcome": {{"type":"INT64","value":0}},
    "batchSize": 100
  }}' | jq
''')
